# Binary Heap

Binary heap is used in:
- heapsort
- implementing priority queue

## Types

1. **Min heap** - highest priority item is assigned lowest value
2. **Max heap** - highest priority item is assigned highest value

## Properties

Binary heap is a **complete binary tree** (stored as an array).

Complete binary tree is a binary tree whose all levels are completely filled except possibly the last level and the last level has to be filled from left to right.

### Array representation:

- Left child of node at index i: `left(i) = 2i + 1`
- Right child of node at index i: `right(i) = 2i + 2`  
- Parent of node at index i: `parent(i) = floor((i - 1)/2)`

### Array representation advantages:

1. Contiguous storage therefore random access
2. Cache friendliness
3. Since it's a complete binary tree, the height is minimum possible height (log n)

## Min Heap

- Complete binary tree
- Every node has value smaller than its descendants

### Main operations:

- constructor (simple)
- insert
- extract min
- decrease key
- delete
- constructor enhanced with build heap

### Utility functions:

- left child
- right child
- parent
- min heapify

![Heap Array to Tree Mapping](images/heap-array-tree.png)

### Building the heap (constructor)

Two ways to turn an unordered array into a heap. Sorting it works but costs
O(n log n). The cheaper way: fix the small subtrees first, then the bigger ones.

Every leaf is already a valid one-element heap, so start at the **last non-leaf
node** -- the parent of the last element, index `(n - 2) // 2` -- and sift down from
there in reverse index order. When `heapify(i)` runs, both of i's subtrees are
already heaps, which is exactly its precondition.

```
[10, 5, 20, 2, 4, 8]      last non-leaf = (6 - 2) // 2 = index 2

i=2:  20 vs child 8          swap  → [10, 5, 8, 2, 4, 20]
i=1:   5 vs children 2, 4    swap  → [10, 2, 8, 5, 4, 20]
i=0:  10 vs children 2, 8    swap  → [2, 10, 8, 5, 4, 20]
      10 keeps sinking             → [2, 4, 8, 5, 10, 20]
```

This is O(n), not O(n log n) -- most nodes are near the bottom and barely move. The
proof is worked out further down the notebook.

**Time:** O(n) &nbsp; **Space:** O(1) plus the recursion in `heapify`

In [ ]:
import math


class MinHeap:
    def __init__(self, ls=None):
        """
        When ls is provided, build a heap out of it.
        Naive approach: Sort the array and then build heap
        Time complexity: O(n log n)

        Efficient approach: find the position of the bottom-most, right-most
        non-leaf node and perform the heapify operation on each non-leaf node in
        reverse level order. The assumption for that node is the left and right
        children are already heapified.
        The last non-leaf node is the parent of the last node,
        i.e. the parent of the node at index (len(ls) - 1),
        i.e. the node at index ((len(ls) - 1) - 1) // 2.

        Time complexity: O(n)
        https://www.geeksforgeeks.org/building-heap-from-array/
        """
        self.arr = [] if ls is None else ls
        i = (len(self.arr) - 2) // 2
        while i >= 0:
            self.heapify(i)
            i -= 1

    def parent(self, i):
        return (i - 1) // 2

    def lchild(self, i):
        return (2 * i) + 1

    def rchild(self, i):
        return (2 * i) + 2


def is_min_heap(arr):
    """True if every node is <= both of its children."""
    n = len(arr)
    return all(
        arr[i] <= arr[child]
        for i in range(n)
        for child in (2 * i + 1, 2 * i + 2)
        if child < n
    )


def test_is_min_heap():
    assert is_min_heap([2, 4, 8, 5, 10, 20]) is True
    assert is_min_heap([]) is True
    assert is_min_heap([1]) is True
    assert is_min_heap([10, 5, 20]) is False  # root larger than its child


test_is_min_heap()

### Heapify (sift down)

Repair a single node that may be too large, **assuming both of its subtrees are
already valid heaps**. Compare it with its two children, swap it with the smaller
child if one of them is smaller, and repeat from the position it moved to. The value
sinks until both children are larger.

That precondition is the thing to remember: `heapify` cannot fix an arbitrarily
scrambled array, only a bad root sitting on top of good subtrees. Every other
operation is arranged to satisfy it.

**Time:** O(log n) -- one comparison pair per level of descent &nbsp;
**Space:** O(log n) recursion depth

In [ ]:
def heapify(self, i):
    """
    Fixes min heap whose root might be violating min heap property

    Time complexity: O(log n)
    Aux space: O(log n)
    """
    arr = self.arr
    lt = self.lchild(i)
    rt = self.rchild(i)
    smallest = i
    n = len(arr)
    if lt < n and arr[lt] < arr[smallest]:
        smallest = lt
    if rt < n and arr[rt] < arr[smallest]:
        smallest = rt
    if smallest != i:
        arr[smallest], arr[i] = arr[i], arr[smallest]
        self.heapify(smallest)


MinHeap.heapify = heapify


def test_create_heap():
    heap = MinHeap([10, 5, 20, 2, 4, 8])
    assert heap.arr == [2, 4, 8, 5, 10, 20]
    assert is_min_heap(heap.arr)
    # build heap only reorders -- it never adds or drops elements
    assert sorted(heap.arr) == [2, 4, 5, 8, 10, 20]

    # empty and single-element heaps
    assert MinHeap().arr == []
    assert MinHeap([7]).arr == [7]

    # a default argument must not leak between instances
    assert MinHeap().arr is not MinHeap().arr

    # already a heap -- unchanged
    assert MinHeap([1, 2, 3]).arr == [1, 2, 3]

    # reverse sorted is the worst case for build heap
    heap = MinHeap([9, 7, 5, 3, 1])
    assert is_min_heap(heap.arr)
    assert heap.arr[0] == 1


test_create_heap()

### Insert (sift up)

Append at the end -- the only O(1) position in an array -- and the tree stays
complete. Then repair the ordering upward: while the new value is smaller than its
parent, swap. It rises until its parent is smaller.

```
insert 1 into [2, 4, 8, 5, 10, 20]

append       [2, 4, 8, 5, 10, 20, 1]     index 6, parent (6-1)//2 = 2 → 8
1 < 8  swap  [2, 4, 1, 5, 10, 20, 8]     index 2, parent 0 → 2
1 < 2  swap  [1, 4, 2, 5, 10, 20, 8]     index 0 → stop
```

Sifting up only ever compares with the parent, never with siblings -- a node smaller
than its parent is automatically smaller than the parent's other child.

**Time:** O(log n) &nbsp; **Space:** O(1)

In [ ]:
def insert(self, x):
    """
    Time complexity: O(log n)

    The idea is to append x to the end of the array - O(1) operation.
    But if it is smaller than its parent, it will violate the min heap property.
    Therefore, we travel the height of the binary heap, and keep swapping x with
    its parent (and grandparent etc.) as needed till it's in its intended place.
    This operation is O(log n) since traveling across the height of a binary
    heap is a log of size operation.
    """
    arr = self.arr
    arr.append(x)
    i = len(arr) - 1
    while i > 0 and arr[self.parent(i)] > arr[i]:
        p = self.parent(i)
        arr[i], arr[p] = arr[p], arr[i]
        i = p


MinHeap.insert = insert


def test_insert():
    heap = MinHeap([2, 4, 8, 5, 10, 20])

    # new minimum bubbles all the way to the root
    heap.insert(1)
    assert heap.arr == [1, 4, 2, 5, 10, 20, 8]
    assert is_min_heap(heap.arr)

    # a large value stays at the bottom
    heap.insert(100)
    assert heap.arr[-1] == 100
    assert is_min_heap(heap.arr)

    # inserting into an empty heap
    heap = MinHeap()
    heap.insert(5)
    assert heap.arr == [5]

    # inserting in increasing order never swaps
    heap = MinHeap()
    for x in [1, 2, 3, 4, 5]:
        heap.insert(x)
    assert heap.arr == [1, 2, 3, 4, 5]

    # inserting in decreasing order swaps every time
    heap = MinHeap()
    for x in [5, 4, 3, 2, 1]:
        heap.insert(x)
    assert heap.arr[0] == 1
    assert is_min_heap(heap.arr)


test_insert()

### Extract min

The minimum is at index 0, but removing index 0 directly would shift every remaining
element -- O(n). The fix is to remove from the **end** instead, where popping is free:

1. Save `arr[0]`, the value to return
2. Move the last element into slot 0 and pop the last slot -- both O(1), and the tree
   stays complete
3. That new root is almost certainly out of place, so `heapify(0)` sinks it

```
[2, 4, 8, 5, 10, 20]        min = 2
last → root   [20, 4, 8, 5, 10]
heapify(0)    [4, 20, 8, 5, 10]     4 is the smaller child
heapify(1)    [4, 5, 8, 20, 10]     5 is the smaller child
```

An empty heap returns `math.inf` as a sentinel rather than raising.

**Time:** O(log n) &nbsp; **Space:** O(log n)

In [ ]:
def extract_min(self):
    """
    Remove min from the heap and use heapify to make sure the min heap property
    holds true for the rest of the array

    Time complexity: O(log n)

    If we remove an element from anywhere other than the last position in an
    array, we'll need to move all other elements and it will be a linear
    operation. To achieve "log n" time, removing the last element is constant
    time, so that's what we do.
        1. swap min with last (constant)
        2. pop last (constant)
        3. then heapify (log n).
    """
    arr = self.arr
    if len(arr) == 0:
        return math.inf
    res = arr[0]
    arr[0] = arr[-1]
    arr.pop()
    self.heapify(0)
    return res


MinHeap.extract_min = extract_min


def test_extract_min():
    heap = MinHeap([10, 5, 20, 2, 4, 8])  # -> [2, 4, 8, 5, 10, 20]
    assert heap.extract_min() == 2
    assert heap.arr == [4, 5, 8, 20, 10]
    assert is_min_heap(heap.arr)

    # repeated extraction yields sorted order -- this is heap sort
    heap = MinHeap([10, 5, 20, 2, 4, 8])
    assert [heap.extract_min() for _ in range(6)] == [2, 4, 5, 8, 10, 20]
    assert heap.arr == []

    # extracting from an empty heap returns infinity as a sentinel
    assert MinHeap().extract_min() == math.inf

    # single element
    heap = MinHeap([7])
    assert heap.extract_min() == 7
    assert heap.arr == []


test_extract_min()

### Decrease key

A key that only gets *smaller* can only break the ordering in one direction --
against its parent. So write the new value and reuse the sift-up loop from `insert`;
the children never need looking at.

This is the operation a priority queue needs when an item already in the queue turns
out to have a better priority. Python's `heapq` has no equivalent, which is why the
Dijkstra notebook pushes a duplicate entry and skips stale pops instead.

**Time:** O(log n) &nbsp; **Space:** O(1)

In [ ]:
def decrease_key(self, i, x):
    """
    Time complexity: O(log n)

    Replace the key at index i with x and then swap it with its parent
    (and grandparent etc.) as needed till it's in its intended place.
    """
    arr = self.arr
    arr[i] = x
    while i != 0 and arr[self.parent(i)] > arr[i]:
        p = self.parent(i)
        arr[p], arr[i] = arr[i], arr[p]
        i = p


MinHeap.decrease_key = decrease_key


def test_decrease_key():
    heap = MinHeap([2, 4, 8, 5, 10, 20])

    # 10 at index 4 becomes 1 and rises to the root
    heap.decrease_key(4, 1)
    assert heap.arr == [1, 2, 8, 5, 4, 20]
    assert is_min_heap(heap.arr)

    # decreasing a value that still exceeds its parent stays put
    heap = MinHeap([2, 4, 8, 5, 10, 20])
    heap.decrease_key(5, 9)  # 20 -> 9, parent is 8
    assert heap.arr == [2, 4, 8, 5, 10, 9]
    assert is_min_heap(heap.arr)

    # decreasing the root is a no-op structurally
    heap = MinHeap([2, 4, 8])
    heap.decrease_key(0, 0)
    assert heap.arr == [0, 4, 8]


test_decrease_key()

### Delete

No new machinery -- compose the two operations you already have. Decrease the target
to `-inf` so it sifts up to the root, then extract the root.

```
delete index 2 from [2, 4, 8, 5, 10, 20]

decrease_key(2, -inf)   [-inf, 4, 2, 5, 10, 20]    -inf rises to the root
extract_min()           [2, 4, 20, 5, 10]          root removed, heap restored
```

Two O(log n) passes instead of one, in exchange for no extra code.

**Time:** O(log n) &nbsp; **Space:** O(log n)

In [ ]:
def delete(self, i):
    """
    Time complexity: O(log n)

    Delete the key at index i by decreasing it to -infinity so it rises to the
    root, then extracting the root.
    """
    if i >= len(self.arr):
        return
    self.decrease_key(i, -math.inf)
    self.extract_min()


MinHeap.delete = delete


def test_delete():
    heap = MinHeap([2, 4, 8, 5, 10, 20])
    heap.delete(2)  # remove the 8
    assert sorted(heap.arr) == [2, 4, 5, 10, 20]
    assert is_min_heap(heap.arr)

    # deleting the root removes the minimum
    heap = MinHeap([2, 4, 8, 5, 10, 20])
    heap.delete(0)
    assert 2 not in heap.arr
    assert heap.arr[0] == 4
    assert is_min_heap(heap.arr)

    # deleting the last element
    heap = MinHeap([2, 4, 8])
    heap.delete(2)
    assert heap.arr == [2, 4]

    # out of range index is ignored
    heap = MinHeap([2, 4, 8])
    heap.delete(10)
    assert heap.arr == [2, 4, 8]

    # deleting every element one by one
    heap = MinHeap([5, 3, 9, 1])
    for _ in range(4):
        heap.delete(0)
    assert heap.arr == []


test_delete()

# Time complexity of build heap operation

Consider the following min heap.

![Min heap](images/min-heap.png)

Maximum nodes at height $h$ can be computed using the following formula:
$n_h = \bigg\lceil \frac {n}{2^{h+1}} \bigg\rceil$

The time required by `heapify` when called on a node with height $h$ is $O(h)$.
Letting $c$ be the constant implicit in asymptotic notation, the total cost can be expressed in the following:
$$= \sum_{h=0}^{\log{n}} \bigg\lceil \frac {n}{2^{h+1}} \bigg\rceil ch$$
Dropping the ceiling gives an upper bound:
$$\leqslant \sum_{h=0}^{\log{n}} \frac {n}{2^{h+1}} \cdot ch$$
Factor out $cn$:
$$= cn \sum_{h=0}^{\log{n}} \frac {h}{2^{h+1}}$$
Rewrite $2^{h+1} = 2 \cdot 2^h$:
$$= cn \sum_{h=0}^{\log{n}} \frac {h}{2 \cdot 2^{h}} = \frac{cn}{2} \sum_{h=0}^{\log{n}} \frac {h}{2^{h}}$$
Since $\frac{1}{2}$ is a constant, this simplifies to:
$$= O\!\left(cn \sum_{h=0}^{\log{n}} \frac {h}{2^{h}}\right)$$
Upperbounding to `∞`, the above can be rewritten as:
$$\leqslant cn \sum_{h=0}^{\infty} \frac {h}{2^{h}}$$
Now it's clear it's an arithmetico-geometric series (derivable by differentiating the geometric series $\sum x^k = \frac{1}{1-x}$ with respect to $x$) with $x = 1/2$, $\sum_{k=0}^{\infty} kx^k = \frac {x}{(1 - x)^2}$
Using the above formula:
$$\leqslant cn * \frac {1/2}{(1 - 1/2)^2} \\\\ = O(n)$$

>
> *Sources*
> Introduction to algorithms 4ed (Cormen et all) - section 6.3
> <https://stackoverflow.com/questions/9755721/how-can-building-a-heap-be-on-time-complexity/62177336#62177336>

# Heap Sort

Can be seen as optimization over selection sort.

In selection sort, we find out the maximum element in the array using linear search, swap it with the last continue with the remaining elements.

The idea of heap sort is, instead of doing a linear search, we maintain the remaining elements in heap structure. With heap data structure, we can extract the maximum or minimum in O(log n) time (finding it is O(1) since it's always at the root) and therefore the overall complexity becomes O(n log n) rather than O(n^2) like selection sort.

## Steps:

1. Build a max heap
2. Repeatedly swap root with the last node, reduce heap size by 1 and heapify

**Time complexity:** O(n log n)  
**Aux space:** O(1) (or O(log n) if we use recursion)

## Notes:

- It's not stable
- Heapsort is 2-3 times slower than quicksort because quicksort has better locality of reference than heapsort
- Used in hybrid sorting algorithms like IntroSort

In [ ]:
def build_heap(arr):
    """Turn arr into a max heap in place. Time: O(n)"""
    n = len(arr)
    for i in range((n - 2) // 2, -1, -1):
        max_heapify(arr, n, i)


def max_heapify(arr, n, i):
    """Sift arr[i] down within the first n elements of arr. Time: O(log n)"""
    largest = i
    left = 2 * i + 1
    right = 2 * i + 2
    if left < n and arr[left] > arr[largest]:
        largest = left
    if right < n and arr[right] > arr[largest]:
        largest = right
    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        max_heapify(arr, n, largest)


def heap_sort(arr):
    """Sort arr in place, ascending. Time: O(n log n), aux space: O(log n)"""
    n = len(arr)
    build_heap(arr)
    for i in range(n - 1, 0, -1):
        arr[i], arr[0] = arr[0], arr[i]  # largest goes to its final position
        max_heapify(arr, i, 0)  # restore the heap over the shrinking prefix


def is_max_heap(arr, n=None):
    """True if the first n elements of arr satisfy the max heap property."""
    n = len(arr) if n is None else n
    return all(
        arr[i] >= arr[child]
        for i in range(n)
        for child in (2 * i + 1, 2 * i + 2)
        if child < n
    )


def test_build_heap():
    arr = [10, 5, 20, 2, 4, 8]
    build_heap(arr)
    assert is_max_heap(arr)
    assert arr[0] == 20  # the maximum ends up at the root
    assert sorted(arr) == [2, 4, 5, 8, 10, 20]


def test_heap_sort():
    arr = [10, 5, 20, 2, 4, 8]
    heap_sort(arr)
    assert arr == [2, 4, 5, 8, 10, 20]

    # duplicates, already sorted, reverse sorted
    arr = [3, 1, 4, 1, 5, 9, 2, 6, 5]
    heap_sort(arr)
    assert arr == [1, 1, 2, 3, 4, 5, 5, 6, 9]

    arr = [1, 2, 3, 4]
    heap_sort(arr)
    assert arr == [1, 2, 3, 4]

    arr = [4, 3, 2, 1]
    heap_sort(arr)
    assert arr == [1, 2, 3, 4]

    # edge cases
    arr = []
    heap_sort(arr)
    assert arr == []

    arr = [1]
    heap_sort(arr)
    assert arr == [1]

    # matches sorted() on random input
    import random

    data = [random.randint(0, 100) for _ in range(50)]
    arr = list(data)
    heap_sort(arr)
    assert arr == sorted(data)


test_build_heap()
test_heap_sort()

# Python Built-in: `heapq` module

Python's `heapq` implements a **min-heap** on a regular `list`.

| Operation | Function | Time |
|-----------|----------|------|
| Push | `heapq.heappush(h, x)` | O(log n) |
| Pop min | `heapq.heappop(h)` | O(log n) |
| Peek min | `h[0]` | O(1) |
| Build heap | `heapq.heapify(h)` | O(n) |
| Push + pop | `heapq.heappushpop(h, x)` | O(log n) |
| N smallest | `heapq.nsmallest(k, iterable)` | O(n log k) |

**No built-in max-heap** -- use the negation trick.

In [ ]:
import heapq

# min-heap operations
h = [10, 5, 20, 2, 4, 8]
heapq.heapify(h)           # O(n) -- [2, 4, 8, 5, 10, 20]
print(h)

heapq.heappush(h, 1)       # O(log n)
print(heapq.heappop(h))    # 1 -- smallest element
print(h[0])                # 2 -- peek without removing

# max-heap via negation trick
nums = [3, 1, 4, 1, 5]
max_h = [-x for x in nums]
heapq.heapify(max_h)
print(-heapq.heappop(max_h))  # 5 -- largest element

# k smallest / k largest
data = [10, 5, 20, 2, 4, 8]
print(heapq.nsmallest(3, data))  # [2, 4, 5]
print(heapq.nlargest(3, data))   # [20, 10, 8]